# 🚀 Telco Customer Churn Prediction — XGBoost Pipeline

This notebook implements an **XGBoost (eXtreme Gradient Boosting)** pipeline for customer churn prediction.

### Key Highlights & Technical Capabilities:
1. **End-to-End Scikit-Learn Pipeline**: Integrated preprocessing (`ChurnCleaner`), high-impact feature engineering (`ChurnFeatureEngineer`), and `XGBClassifier`.
2. **Handling Class Imbalance**: Automatic computation and integration of `scale_pos_weight` (negative / positive class ratio ~ 2.77) to prioritize churn recall without sacrificing precision.
3. **5-Fold Stratified Cross-Validation**: Rigorous out-of-fold (OOF) evaluation to avoid data leakage.
4. **Out-of-Fold Decision Threshold Tuning**: Optimizes the probability threshold for F1-score / business ROI.
5. **Evaluation Suite**: ROC-AUC, PR-AUC, Confusion Matrix, Classification Report, and ROC/PR Curves.
6. **Interpretability & Feature Importance**: In-depth analysis of feature importance across *Gain* (predictive power) and *Weight* (frequency), plus individual customer prediction breakdown.
7. **Pipeline Persistence**: Serializes the final trained pipeline to `pipeline_xgb.pkl`.


In [ ]:
import sys
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score,
    average_precision_score
)

import xgboost as xgb
from xgboost import XGBClassifier

# Add custom library paths
sys.path.append(os.path.abspath("../dataset and other libs"))
from churn_feature_engineering import ChurnCleaner, ChurnFeatureEngineer
from trace_path import trace_customer_path

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f"XGBoost version: {xgb.__version__}")


## 1. Load Dataset & Stratified Split
We load the raw Telco customer dataset and perform a stratified 80/20 train/test split.


In [ ]:
# Load raw dataset using robust path resolution
data_path = os.path.abspath("../dataset and other libs/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = pd.read_csv(data_path)

# Separate features (X) and target (y)
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset Dimensions: {df.shape}")
print(f"Training samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")
print("Churn Distribution in Training Data:")
print(y_train.value_counts(normalize=True).round(3))


## 2. Compute `scale_pos_weight` for Class Imbalance
In the Telco churn dataset, ~26.5% of customers churn and ~73.5% stay.
Gradient boosting models can naturally bias toward majority classes unless balanced.
XGBoost provides `scale_pos_weight = count(negative) / count(positive)` to scale gradients for positive (churn) cases.


In [ ]:
num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
scale_pos_weight = num_neg / num_pos

print(f"Negative (Retained) samples: {num_neg}")
print(f"Positive (Churned) samples:  {num_pos}")
print(f"Calculated scale_pos_weight: {scale_pos_weight:.3f}")


## 3. Build End-to-End XGBoost Pipeline
We assemble a scikit-learn `Pipeline` chaining:
1. `ChurnCleaner`: Handles type conversions, missing total charges, and clean one-hot encoding.
2. `ChurnFeatureEngineer`: Adds engineered features (`ChargeDiff`, `AvgMonthlyCharges`, `TotalServices`, `HighRisk_Contract_Payment`, etc.).
3. `XGBClassifier`: Tuned gradient boosting tree classifier with tree regularization, shrinkage (`learning_rate=0.05`), and subsampling.


In [ ]:
pipeline_xgb = Pipeline([
    ('cleaner', ChurnCleaner()),
    ('feature_engineer', ChurnFeatureEngineer()),
    ('classifier', XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=3,
        gamma=0.2,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])

pipeline_xgb


## 4. 5-Fold Stratified Cross-Validation & Out-of-Fold (OOF) Calibration
To avoid data leakage, we perform 5-fold cross-validation exclusively on the training set to evaluate generalization and find the optimal decision threshold for F1-score.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = cross_val_predict(pipeline_xgb, X_train, y_train, cv=cv, method='predict_proba')[:, 1]

oof_auc = roc_auc_score(y_train, oof_probs)
oof_pr_auc = average_precision_score(y_train, oof_probs)
print(f"5-Fold CV OOF ROC-AUC: {oof_auc:.4f}")
print(f"5-Fold CV OOF PR-AUC:  {oof_pr_auc:.4f}")

# Precision, Recall, and F1 across thresholds
precisions, recalls, thresholds = precision_recall_curve(y_train, oof_probs)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)

best_idx = np.argmax(f1_scores[:-1])
best_thresh = thresholds[best_idx]

print(f"Optimal OOF Decision Threshold: {best_thresh:.3f}")
print(f"OOF F1-Score at optimal threshold: {f1_scores[best_idx]:.4f}")
print(f"OOF Precision: {precisions[best_idx]:.4f} | Recall: {recalls[best_idx]:.4f}")

# Plot Threshold Calibration Curves
plt.figure(figsize=(9, 5))
plt.plot(thresholds, precisions[:-1], label='Precision', color='#1f77b4', lw=2)
plt.plot(thresholds, recalls[:-1], label='Recall', color='#ff7f0e', lw=2)
plt.plot(thresholds, f1_scores[:-1], label='F1-Score', color='#2ca02c', lw=2)
plt.axvline(best_thresh, color='#d62728', linestyle='--', lw=2, label=f'Optimal Thresh ({best_thresh:.2f})')
plt.title('Out-Of-Fold Precision, Recall & F1 vs. Decision Threshold (XGBoost)', fontsize=13)
plt.xlabel('Probability Threshold', fontsize=11)
plt.ylabel('Score', fontsize=11)
plt.legend(loc='lower center', frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5. Train on Full Training Set & Evaluate on Test Set
Now we train the pipeline on the full training split `X_train` and test once on the untouched `X_test` using our OOF-calibrated decision threshold.


In [ ]:
# Fit pipeline on entire training data
pipeline_xgb.fit(X_train, y_train)

# Predict probabilities and apply optimal threshold
y_prob_test = pipeline_xgb.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= best_thresh).astype(int)

# Metrics
test_auc = roc_auc_score(y_test, y_prob_test)
test_pr_auc = average_precision_score(y_test, y_prob_test)
cm = confusion_matrix(y_test, y_pred_test)

print("=" * 48)
print("       XGBOOST TEST SET PERFORMANCE EVALUATION")
print("=" * 48)
print(f"ROC-AUC Score:          {test_auc:.4f}")
print(f"PR-AUC (Avg Precision): {test_pr_auc:.4f}")
print(f"Decision Threshold:     {best_thresh:.3f}")
print("
Confusion Matrix:")
print(cm)
print("
Classification Report:")
print(classification_report(y_test, y_pred_test, target_names=['Retained (0)', 'Churned (1)']))


## 6. Evaluation Visualizations: Confusion Matrix, ROC & PR Curves


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Predicted Retained', 'Predicted Churn'],
            yticklabels=['Actual Retained', 'Actual Churn'])
axes[0].set_title('Confusion Matrix', fontsize=12)

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
axes[1].plot(fpr, tpr, color='#0066cc', lw=2.5, label=f'XGBoost (AUC = {test_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', alpha=0.7)
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontsize=12)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right', frameon=True)
axes[1].grid(True, alpha=0.3)

# 3. Precision-Recall Curve
pr_prec, pr_rec, _ = precision_recall_curve(y_test, y_prob_test)
axes[2].plot(pr_rec, pr_prec, color='#e65100', lw=2.5, label=f'XGBoost (PR-AUC = {test_pr_auc:.4f})')
axes[2].axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.7, label=f'Baseline ({y_test.mean():.2f})')
axes[2].set_title('Precision-Recall Curve', fontsize=12)
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend(loc='lower left', frameon=True)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 7. XGBoost Feature Importance (Gain vs. Weight)
XGBoost provides multiple feature importance metrics:
- **Gain**: The average training loss reduction brought by splits using this feature (most direct measure of predictive power).
- **Weight**: The frequency with which a feature appears in splits across all trees.


In [ ]:
model = pipeline_xgb.named_steps['classifier']
feature_names = pipeline_xgb.named_steps['feature_engineer'].get_feature_names_out()

# Get Booster object and feature score mappings
booster = model.get_booster()
score_gain = booster.get_score(importance_type='gain')
score_weight = booster.get_score(importance_type='weight')

# Map f0, f1... to real feature names if needed
mapped_gain = {}
mapped_weight = {}
for i, name in enumerate(feature_names):
    f_key = f'f{i}'
    if f_key in score_gain:
        mapped_gain[name] = score_gain[f_key]
        mapped_weight[name] = score_weight.get(f_key, 0)
    elif name in score_gain:
        mapped_gain[name] = score_gain[name]
        mapped_weight[name] = score_weight.get(name, 0)

importance_df = pd.DataFrame({
    'Feature': list(mapped_gain.keys()),
    'Gain': list(mapped_gain.values()),
    'Weight': [mapped_weight.get(k, 0) for k in mapped_gain.keys()]
}).sort_values(by='Gain', ascending=False).reset_index(drop=True)

print("=== Top 15 Most Predictive Features by Gain ===")
print(importance_df.head(15).to_string(index=False))

# Plot Top Features
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

top15_gain = importance_df.head(15).iloc[::-1]
ax1.barh(top15_gain['Feature'], top15_gain['Gain'], color='#2b5c8f')
ax1.set_title('Top 15 Features by Average Gain (Predictive Impact)', fontsize=12)
ax1.set_xlabel('Gain (Split Loss Improvement)')
ax1.grid(True, alpha=0.3)

top15_weight = importance_df.sort_values(by='Weight', ascending=False).head(15).iloc[::-1]
ax2.barh(top15_weight['Feature'], top15_weight['Weight'], color='#00897b')
ax2.set_title('Top 15 Features by Split Count (Weight)', fontsize=12)
ax2.set_xlabel('Frequency / Split Count')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Customer-Level Prediction & Risk Inspection
We inspect single customer predictions, checking probability scores and risk classification.


In [ ]:
# Sample 5 customers from test set
sample_raw = X_test.head(5)
sample_probs = pipeline_xgb.predict_proba(sample_raw)[:, 1]
sample_preds = (sample_probs >= best_thresh).astype(int)
sample_actual = y_test.head(5).values

sample_results = pd.DataFrame({
    'Actual Churn': sample_actual,
    'Predicted Churn': sample_preds,
    'Churn Probability': sample_probs.round(4),
    'Risk Level': ['High' if p >= 0.7 else 'Medium' if p >= best_thresh else 'Low' for p in sample_probs]
})

print("=== Sample Customer Inference ===")
display(sample_results) if 'display' in dir() else print(sample_results)


## 9. Save Trained XGBoost Pipeline
Save the complete pipeline to a pickle file for production deployment, Streamlit app, or batch scoring.


In [ ]:
output_model_path = 'pipeline_xgb.pkl'
joblib.dump(pipeline_xgb, output_model_path)
print(f"Successfully serialized and saved XGBoost pipeline to: {output_model_path}")
print(f"File size: {os.path.getsize(output_model_path) / 1024:.2f} KB")
